# Using ras_commander to manage Lava Barriers in HEC-RAS project

This notework will show you how to get started using ras_commander library for HEC-RAS and how to manage lava barriers in Python.

## How to get started

RAS Commander (ras_commander) is a Python library for automating HEC-RAS operations, providing a set of tools to interact with HEC-RAS project files, execute simulations, and manage project data.

Before using ras_commander in Python, you must:

- Install HEC-RAS on your computer.

- Use the HEC-RAS GUI to create a project (.prj) with these store in the same project folder (in this tutorial my folder name is "barrier_tutorial"):

  - At least one geometry (.g## file)
  - At least one plan (.p## file)
  - Unsteady flow data (.u## file)
If following Ed's video to create a HEC-RAS project, you should already have these files in your computer.

- Now fork this repository https://github.com/gpt-cmdr/ras-commander.git to your GitHub account and computer. Now you should have a ras-commander folder in your computer, this folder stores all the functions and Python setup required to use ras_commander library. Have a quick read through the README file of this repository to better understand it

- Choose an AI RAS Commander Assistant on repository page and ask it to help you install ras_commander library in Jupyter Notebook of VSCode or Anaconda easily (I used RAS Commander Library Assistant on ChatGPT).

- After successfully installing ras_commander library, you should be able to run this notebook in the environment that has ras_commander installed.

- Note: Sometimes the AI Assistant has not been updated with the functions of ras_commander yet, so I also recommend looking at the 'examples' and 'ras_commander' folder in the 'ras-commander' folder you have forked to know how to use the functions.

- Most of these tutorials are based on "02_plan_and_geometry_operations.ipynb" in the 'examples' folder in 'ras-commander'.

## Initialise project

In [220]:
from pathlib import Path
from ras_commander import init_ras_project, ras, RasUtils, HdfStruc, HdfResultsPlan, RasPlan, RasCmdr
import pandas as pd
#from IPython import display

# Replace with your path to your HEC-RAS project folder
project_path = Path(r"C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial")
ras_exe = Path(r"C:\Program Files (x86)\HEC\HEC-RAS\6.7 Beta 4\Ras.exe") # replace with your path to Ras.exe

init_ras_project(project_path, ras_version=ras_exe)

2025-10-13 00:04:46 - ras_commander.RasMap - INFO - Successfully parsed RASMapper file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.rasmap


## Manage SA/2D connection as the barrier
The code below assumes that you have an SA/2D connection as a weir/embankment created in HEC-RAS GUI. Watch the video on how to create this if you haven't.

In [185]:
# Read the Geometry HDF file that has SA/2D connection
geom_hdf = r"C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf" # replace with your path to .g##.hdf
structures = HdfStruc.get_structures(geom_hdf)
structures.head()

2025-10-12 22:31:14 - ras_commander.HdfStruc - INFO - Using HDF file from direct string path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf
2025-10-12 22:31:14 - ras_commander.HdfStruc - INFO - Final validated HDF file path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf
2025-10-12 22:31:14 - ras_commander.HdfBase - INFO - Using existing Path object HDF file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf
2025-10-12 22:31:14 - ras_commander.HdfBase - INFO - Final validated HDF file path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf
2025-10-12 22:31:14 - ras_commander.HdfBase - INFO - Found projection in HDF file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf
2025-10-12 22:31:15 - ras_commander.HdfStruc - INFO - Successfully extracted st

,Type,Mode,River,Reach,RS,Connection,Groupname,US Type,US River,US Reach,...,US XS Mann (Count),US BR Mann (Index),US BR Mann (Count),DS XS Mann (Index),DS XS Mann (Count),DS BR Mann (Index),DS BR Mann (Count),RC (Index),RC (Count),Profile_Data
0,Connection,Weir/Gate/Culverts,,,,SA2D Conn 1,"Perimeter 1, SA2D Conn 1",2D,,,...,0,0,0,0,0,0,0,0,0,"[{'Station': 0.0, 'Elevation': 139.69000244140..."


In [3]:
# Print the Profile Data of the weir/embankment
profile_data = structures.iloc[0]["Profile_Data"] # choose the Profile data of the first row, change this if you have multiple structures
base_df = pd.DataFrame(profile_data)
base_df

,Station,Elevation
0,0.000000,139.690002
1,1.200000,139.649994
2,4.800000,139.789993
3,6.400000,139.630005
4,9.500000,139.550003
...,...,...
445,591.599976,138.479996
446,592.099976,138.009995
447,592.599976,138.029999
448,593.599976,138.880005


In [23]:
# Raise elevation values of the barrier
variant_sheets = {}  # dict of sheet names

for delta in range(1, 8):  # Add 1 to 7 metres to barrier height
    df_var = base_df.copy()
    df_var["Elevation"] = df_var["Elevation"] + float(delta)

    sheet_name = f"+{delta}m"
    variant_sheets[sheet_name] = df_var

In [25]:
# Write modified profile output to Excel file
# Replace with your output path
output_xlsx = Path(r"C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\weir_profiles_variants.xlsx")
engine = None  # or "xlsxwriter"
try:
    with pd.ExcelWriter(output_xlsx, engine=engine) as xw:
        # Write each scenario as its own sheet
        for sheet, df_out in variant_sheets.items():
            # Excel sheet names max length is 31
            safe_sheet = sheet[:31]
            df_out.to_excel(xw, index=False, sheet_name=safe_sheet)
        
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Excel writer engine not found. Install one of:\n"
        "  pip install openpyxl   # recommended for .xlsx\n"
        "  pip install XlsxWriter # alternative engine"
    ) from exc

print("Done. Check the sheets created)

- Now you've just created an Excel file where each sheet stores the Station-Elevation values of modified heights.
- Rascommander cannot modify the geometry file directly so we need to copy and paste these values to the new geometry files in HEC-RAS GUI. Before this, let's quickly clone the current plan and geometry file to new files.

## Clone plan and geometry

In [38]:
# Read the plan files of this project
print("\nHEC-RAS Project Plan Data:")
ras.plan_df


HEC-RAS Project Plan Data:


,plan_number,unsteady_number,geometry_number,Plan Title,Program Version,Short Identifier,Simulation Date,Computation Interval,Mapping Interval,Run HTab,...,DSS File,Friction Slope Method,UNET D2 SolverType,UNET D2 Name,HDF_Results_Path,Geom File,Geom Path,Flow File,Flow Path,full_path
0,01,01,01,plan 1,6.70,101,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
1,02,01,02,plan 2,6.70,102,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,02,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
2,03,01,03,plan 3,6.70,103,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,03,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
3,04,01,04,plan 4,6.70,104,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,04,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
4,05,01,05,plan 5,6.70,105,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,05,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
5,06,01,06,plan 6,6.70,106,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,06,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
6,07,01,07,plan 7,6.70,107,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,07,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
7,08,01,08,plan 8,6.70,108,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,08,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...


In [5]:
# Clone plan "01" to create 7 new plans
# RUNNING THIS WILL CREATE NEW PLANS, ONLY RUN ONCE UNLESS YOU WANT TO CREATE MORE NEW PLANS!!!
for i in range(2, 9):
    plan_shortid = str(100 + i)
    new_plan_number = RasPlan.clone_plan("01", new_plan_shortid=plan_shortid)
    print(f"New plan created: {new_plan_number}")

2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p02
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - Successfully updated file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p02
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - Project file updated with new Plan entry: 02
2025-09-30 23:30:01 - ras_commander.RasMap - INFO - Successfully parsed RASMapper file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.rasmap
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p03
2025-09-30 23:30:01 - r

New plan created: 02
New plan created: 03
New plan created: 04


2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - Project file updated with new Plan entry: 05
2025-09-30 23:30:01 - ras_commander.RasMap - INFO - Successfully parsed RASMapper file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.rasmap
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p06
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - Successfully updated file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p06
2025-09-30 23:30:01 - ras_commander.RasUtils - INFO - Project file updated with new Plan entry: 06
2025-09-30 23:30:01 - ras_commander.RasMap - INFO - Successfully parsed RASMapper file: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.rasmap


New plan created: 05
New plan created: 06
New plan created: 07


In [14]:
# Update the new plans' titles
for i in range(2, 9):
    new_title = f"plan {i}"
    plan_number = f"0{i}"
    RasPlan.set_plan_title(plan_number, new_title=new_title)
    
    print(f"\nUpdated plan title: {RasPlan.get_plan_title(plan_number)}")

2025-09-30 23:45:27 - ras_commander.RasUtils - INFO - Constructed plan file path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p02
2025-09-30 23:45:27 - ras_commander.RasPlan - INFO - Updated Plan Title in plan file to: plan 2
2025-09-30 23:45:27 - ras_commander.RasPlan - INFO - Retrieved Plan Title: plan 2
2025-09-30 23:45:27 - ras_commander.RasUtils - INFO - Constructed plan file path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p03
2025-09-30 23:45:27 - ras_commander.RasPlan - INFO - Updated Plan Title in plan file to: plan 3
2025-09-30 23:45:27 - ras_commander.RasPlan - INFO - Retrieved Plan Title: plan 3
2025-09-30 23:45:27 - ras_commander.RasUtils - INFO - Constructed plan file path: C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.p04
2025-09-30 23:45:27 - ras_commander.RasPlan - INFO - Updated Plan Title in plan file to: plan 4
2025-09-30 23:45:27 - r


Updated plan title: plan 2

Updated plan title: plan 3

Updated plan title: plan 4

Updated plan title: plan 5

Updated plan title: plan 6

Updated plan title: plan 7


In [12]:
# Check the plans again
print("\nHEC-RAS Project Plan Data:")
ras.plan_df


HEC-RAS Project Plan Data:


,plan_number,unsteady_number,geometry_number,Plan Title,Program Version,Short Identifier,Simulation Date,Computation Interval,Mapping Interval,Run HTab,...,DSS File,Friction Slope Method,UNET D2 SolverType,UNET D2 Name,HDF_Results_Path,Geom File,Geom Path,Flow File,Flow Path,full_path
0,01,01,01,plan 1,6.70,101,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
1,02,01,01,plan 2,6.70,102,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
2,03,01,01,plan 3,6.70,103,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
3,04,01,01,plan 4,6.70,104,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
4,05,01,01,plan 5,6.70,105,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
5,06,01,01,plan 6,6.70,106,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
6,07,01,01,plan 7,6.70,107,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
7,08,01,01,plan 1,6.70,108,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,01,None,01,None,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...


In [13]:
# Display the geometry files
print("Geometry files:")
ras.geom_df

Geometry files:


,geom_file,geom_number,full_path,hdf_path
0,g01,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...


In [14]:
# Clone geometry "01" to create 7 new geometry files
for i in range(7):
    new_geom_number = RasPlan.clone_geom("01")
    print(f"New geometry created: {new_geom_number}")

2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g02
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g02.hdf
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - Project file updated with new Geom entry: 02
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g03
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb

New geometry created: 02
New geometry created: 03
New geometry created: 04
New geometry created: 05
New geometry created: 06


2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g07.hdf
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - Project file updated with new Geom entry: 07
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01 to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g08
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - File cloned from C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g01.hdf to C:\Users\Phuong\OneDrive\Documents\unimelb\Lava intern\barrier_tutorial\barrier_tutorial.g08.hdf
2025-10-11 21:31:22 - ras_commander.RasUtils - INFO - Project file updated with new Geom entry: 08


New geometry created: 07
New geometry created: 08


In [39]:
# Display updated geometry files
print("\nUpdated geometry files:")
ras.geom_df


Updated geometry files:


,geom_file,geom_number,full_path,hdf_path
0,g01,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
1,g02,02,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
2,g03,03,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
3,g04,04,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
4,g05,05,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
5,g06,06,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
6,g07,07,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
7,g08,08,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...


In [16]:
# Set the new geometry files for the cloned plans
for i in range(2, 8):
    new_plan_number = f"0{i}"
    new_geom_number = f"0{i}"
    updated_geom_df = RasPlan.set_geom(new_plan_number, new_geom_number)
    print(f"Updated geometry for plan {new_plan_number} to geometry {new_geom_number}")

2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom File in plan file to g02 for plan 02
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Geometry for plan 02 set to 02
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom File in plan file to g03 for plan 03
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Geometry for plan 03 set to 03
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom File in plan file to g04 for plan 04
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Geometry for plan 04 set to 04
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom File in plan file to g05 for plan 05
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Geometry for plan 05 set to 05
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom File in plan file to g06 for plan 06
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Geometry for plan 06 set to 06
2025-10-11 21:37:03 - ras_commander.RasPlan - INFO - Updated Geom

Updated geometry for plan 02 to geometry 02
Updated geometry for plan 03 to geometry 03
Updated geometry for plan 04 to geometry 04
Updated geometry for plan 05 to geometry 05
Updated geometry for plan 06 to geometry 06
Updated geometry for plan 07 to geometry 07


In [187]:
# Check the plans again
print("\nHEC-RAS Project Plan Data:")
ras.plan_df


HEC-RAS Project Plan Data:


,plan_number,unsteady_number,geometry_number,Plan Title,Program Version,Short Identifier,Simulation Date,Computation Interval,Mapping Interval,Run HTab,...,DSS File,Friction Slope Method,UNET D2 SolverType,UNET D2 Name,HDF_Results_Path,Geom File,Geom Path,Flow File,Flow Path,full_path
0,01,01,01,plan 1,6.70,101,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
1,02,01,02,plan 2,6.70,102,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,02,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
2,03,01,03,plan 3,6.70,103,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,03,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
3,04,01,04,plan 4,6.70,104,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,04,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
4,05,01,05,plan 5,6.70,105,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,05,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
5,06,01,06,plan 6,6.70,106,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,06,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
6,07,01,07,plan 7,6.70,107,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,None,07,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...
7,08,01,08,plan 8,6.70,108,"01SEP2008,,05SEP2008,",1MIN,1HOUR,-1,...,dss,1,PARDISO (Direct),Perimeter 1,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,08,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,01,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...,C:\Users\Phuong\OneDrive\Documents\unimelb\Lav...


- You should now see that the new geometry files are associated with the new plans.
- Now let's go back to HEC-RAS GUI, open each plan and change the station-elevation data of each plan.